# Time Series Course — m10-fl2 (block 2)

_From the **Time Series & Forecasting** course on Zylo._

Run all cells (`Runtime → Run all` or `Ctrl+F9`) to execute.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Generate synthetic NSE Nifty-like daily returns
np.random.seed(42)
n_days = 2000
daily_returns = np.random.normal(0.0003, 0.012, n_days)  # slight upward drift
# Add momentum: positive days slightly more likely after positive streaks
for i in range(5, n_days):
    if np.mean(daily_returns[i-5:i]) > 0:
        daily_returns[i] += 0.001  # momentum boost

prices = 15000 * np.exp(np.cumsum(daily_returns))  # Nifty-like starting level

# Binary labels: 1 = price goes up tomorrow, 0 = down
labels = (daily_returns[1:] > 0).astype(np.float32)
features = prices[:-1]  # use prices to predict next-day direction

# Normalize
train_size = 1600
train_mean = features[:train_size].mean()
train_std = features[:train_size].std()
features_norm = (features - train_mean) / train_std

# Windowing
lookback = 60
X, y = [], []
for i in range(lookback, len(features_norm)):
    X.append(features_norm[i-lookback:i])
    y.append(labels[i-1])  # direction of next day
X = np.array(X)
y = np.array(y)

split = train_size - lookback
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

class DirectionLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(1, 32, num_layers=1, batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(32, 16), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(16, 1), nn.Sigmoid()
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

X_tr_t = torch.FloatTensor(X_train).unsqueeze(-1)
y_tr_t = torch.FloatTensor(y_train).unsqueeze(-1)
X_te_t = torch.FloatTensor(X_test).unsqueeze(-1)
y_te_t = torch.FloatTensor(y_test).unsqueeze(-1)

model = DirectionLSTM()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss()
loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=32, shuffle=True)

for epoch in range(30):
    model.train()
    for xb, yb in loader:
        pred = model(xb); loss = criterion(pred, yb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()

model.eval()
with torch.no_grad():
    test_probs = model(X_te_t).numpy().flatten()
    test_preds = (test_probs > 0.5).astype(int)
    accuracy = (test_preds == y_test).mean()
    print(f"Direction accuracy: {accuracy:.1%}")
    print(f"Always-up baseline: {y_test.mean():.1%}")
    print(f"Edge over baseline: {accuracy - y_test.mean():.1%}")